## 1. Setup and Data Loading
Load the protein sequence dataset for K-fold cross-validation training.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
from contextlib import nullcontext

seq_df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv', usecols=['pdb_id','seq'])
lab_df = pd.read_csv('data/2018-06-06-ss.cleaned.csv', usecols=['pdb_id','sst8','sst3'])
df = pd.merge(seq_df, lab_df, on='pdb_id', how='inner')
df['seq'] = df['seq'].str.replace('*','X')
df = df.dropna(subset=['seq','sst8','sst3']).copy()
df = df[(df['seq'].str.len()==df['sst8'].str.len()) & (df['seq'].str.len()==df['sst3'].str.len())].reset_index(drop=True)
df['len'] = df['seq'].str.len()
print(df.head()); df.info()

## 2. Create Vocabularies

In [ ]:
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}
all_chars = set(''.join(df['seq']))
seq_vocab = {char: i+1 for i, char in enumerate(sorted(list(all_chars)))}
seq_vocab['<pad>'] = 0
vocab_size = len(seq_vocab)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size

## 3. Dataset and Collate

In [ ]:
class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences, sst8_labels, sst3_labels, seq_vocab, ss8_vocab, ss3_vocab):
        self.sequences = sequences
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels
        self.seq_vocab = seq_vocab
        self.ss8_vocab = ss8_vocab
        self.ss3_vocab = ss3_vocab
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        ss8 = self.sst8_labels[idx]
        ss3 = self.sst3_labels[idx]
        seq_tokens = [self.seq_vocab.get(c, 0) for c in seq]
        ss8_tokens = [self.ss8_vocab.get(c, -1) for c in ss8][:len(seq_tokens)]
        ss3_tokens = [self.ss3_vocab.get(c, -1) for c in ss3][:len(seq_tokens)]
        return torch.tensor(seq_tokens, dtype=torch.long), torch.tensor(ss8_tokens, dtype=torch.long), torch.tensor(ss3_tokens, dtype=torch.long)

def collate_fn(batch):
    seqs, ss8s, ss3s = zip(*batch)
    padded_seqs = pad_sequence(seqs, batch_first=True, padding_value=seq_vocab['<pad>'])
    padded_ss8s = pad_sequence(ss8s, batch_first=True, padding_value=-1)
    padded_ss3s = pad_sequence(ss3s, batch_first=True, padding_value=-1)
    return padded_seqs, padded_ss8s, padded_ss3s

## 4. K-Fold Cross-Validation Setup
Prepare data splits for K-fold cross-validation training.

In [ ]:
from sklearn.model_selection import KFold

# K-Fold Cross-Validation Setup: Split data into train+val (90%) and test (10%)
n = len(df)
n_test = int(n * 0.1)
test_indices = list(range(n - n_test, n))
train_val_indices = list(range(n - n_test))

# Test dataset
test_dataset = ProteinSequenceDataset(df.iloc[test_indices]['seq'].tolist(), df.iloc[test_indices]['sst8'].tolist(), df.iloc[test_indices]['sst3'].tolist(), seq_vocab, ss8_vocab, ss3_vocab)

# K-Fold configuration
n_folds = 5
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

embedding_dim = 128

print(f'K-Fold Setup:')
print(f'  Train+Val: {len(train_val_indices)} samples (for K-fold cross-validation)')
print(f'  Test: {len(test_dataset)} samples (held-out)')
print(f'  Embedding dimension: {embedding_dim}')


## 5. RNN Model (learned embeddings)

In [ ]:
class ProteinRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, layers=2, dropout=0.3):
        super().__init__()
        self.pad_idx = seq_vocab['<pad>']
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=self.pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=layers, bidirectional=False, batch_first=True, dropout=dropout if layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.q8_head = nn.Linear(hidden_dim, 8)
        self.q3_head = nn.Linear(hidden_dim, 3)
    def forward(self, x, lengths=None):
        emb = self.embedding(x)
        if lengths is None:
            lengths = (x != self.pad_idx).sum(dim=1).to(torch.int64).cpu()
        orig_len = emb.size(1)
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        packed_out,_ = self.lstm(packed)
        x,_ = pad_packed_sequence(packed_out, batch_first=True, total_length=orig_len)
        x = self.dropout(x)
        return self.q8_head(x), self.q3_head(x)

model = ProteinRNN(vocab_size=vocab_size, embed_dim=embedding_dim).to(device)
if torch.cuda.device_count()>1:
    model = nn.DataParallel(model)
model

## 6. Training Loop (with SOV) - K-Fold Cross-Validation

**Training Configuration:**
- **K-Fold Cross-Validation**: 5 folds
- **Epochs per fold**: 50
- **Early Stopping**: Patience = 5 epochs
- **Model**: RNN (LSTM)
- **Embedding Type**: Learned from scratch (no pre-trained embeddings)

In [ ]:
from torch.utils.data import Subset

def compute_accuracy(pred_logits, labels):
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

q3_id_to_char = {0: 'H', 1: 'E', 2: 'C'}

def get_segments(sequence_chars, state):
    segments = []
    start = -1
    for i, char in enumerate(sequence_chars):
        if char == state:
            if start == -1: start = i
        elif start != -1:
            segments.append((start, i - 1)); start = -1
    if start != -1: segments.append((start, len(sequence_chars) - 1))
    return segments

def compute_sov_q3(pred_logits, labels):
    preds = pred_logits.argmax(-1)
    batch_size = preds.shape[0]
    batch_sov_score = 0.0
    for i in range(batch_size):
        pred_seq = preds[i]; true_seq = labels[i]
        mask = true_seq != -1
        pred_seq_filtered = pred_seq[mask]; true_seq_filtered = true_seq[mask]
        if len(true_seq_filtered) == 0: continue
        pred_chars = [q3_id_to_char.get(pid.item(), 'C') for pid in pred_seq_filtered]
        true_chars = [q3_id_to_char.get(tid.item(), 'C') for tid in true_seq_filtered]
        total_weighted_sov = 0.0; total_residues = 0.0
        for state in ['H','E','C']:
            true_segments = get_segments(true_chars, state)
            pred_segments = get_segments(pred_chars, state)
            state_residues = sum(1 for char in true_chars if char == state)
            total_residues += state_residues
            if not true_segments: continue
            for obs_start, obs_end in true_segments:
                len_obs = (obs_end - obs_start + 1)
                best_min_ov, best_max_ov, best_len_pred = 0, len_obs, 0
                for pred_start, pred_end in pred_segments:
                    overlap_start = max(obs_start, pred_start)
                    overlap_end = min(obs_end, pred_end)
                    min_ov = max(0, overlap_end - overlap_start + 1)
                    if min_ov > 0:
                        max_ov = max(obs_end, pred_end) - min(obs_start, pred_start) + 1
                        len_pred = (pred_end - pred_start + 1)
                        if min_ov > best_min_ov:
                            best_min_ov, best_max_ov, best_len_pred = min_ov, max_ov, len_pred
                if best_min_ov > 0:
                    delta = min(best_max_ov - best_min_ov, best_min_ov, len_obs // 2, best_len_pred // 2)
                    segment_sov = (best_min_ov + delta) / best_max_ov
                else:
                    segment_sov = 0.0
                total_weighted_sov += (segment_sov * len_obs)
        if total_residues > 0:
            batch_sov_score += (total_weighted_sov / total_residues)
    return batch_sov_score / batch_size

# K-Fold Cross-Validation
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)

num_epochs = 50
patience = 5
fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx + 1}/{n_folds}")
    print(f"{'='*60}")
    
    # Create datasets for this fold
    train_fold_indices = [train_val_indices[i] for i in train_idx]
    val_fold_indices = [train_val_indices[i] for i in val_idx]
    
    train_dataset = ProteinSequenceDataset(df.iloc[train_fold_indices]['seq'].tolist(), df.iloc[train_fold_indices]['sst8'].tolist(), df.iloc[train_fold_indices]['sst3'].tolist(), seq_vocab, ss8_vocab, ss3_vocab)
    val_dataset = ProteinSequenceDataset(df.iloc[val_fold_indices]['seq'].tolist(), df.iloc[val_fold_indices]['sst8'].tolist(), df.iloc[val_fold_indices]['sst3'].tolist(), seq_vocab, ss8_vocab, ss3_vocab)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
    
    print(f"Fold {fold_idx + 1} - Train: {len(train_dataset)}, Val: {len(val_dataset)}")
    
    # Initialize model for this fold
    model = ProteinRNN(vocab_size=vocab_size, embed_dim=embedding_dim).to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    best_val_acc_q8 = 0.0
    epochs_no_improve = 0
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = train_acc_q8 = train_acc_q3 = train_sov_q3 = 0.0
        for seqs, ss8, ss3 in tqdm(train_loader, desc=f'Fold {fold_idx+1} Epoch {epoch+1}/{num_epochs}'):
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            lengths = (seqs != seq_vocab['<pad>']).sum(dim=1).to(torch.int64).cpu()
            optimizer.zero_grad()
            q8_logits, q3_logits = model(seqs, lengths=lengths)
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_acc_q8 += compute_accuracy(q8_logits, ss8)
            train_acc_q3 += compute_accuracy(q3_logits, ss3)
            train_sov_q3 += compute_sov_q3(q3_logits, ss3)
        
        train_loss /= len(train_loader)
        train_acc_q8 /= len(train_loader)
        train_acc_q3 /= len(train_loader)
        train_sov_q3 /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = val_acc_q8 = val_acc_q3 = val_sov_q3 = 0.0
        with torch.no_grad():
            for seqs, ss8, ss3 in val_loader:
                seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
                lengths = (seqs != seq_vocab['<pad>']).sum(dim=1).to(torch.int64).cpu()
                q8_logits, q3_logits = model(seqs, lengths=lengths)
                loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
                loss = loss_q8 + 0.5 * loss_q3
                val_loss += loss.item()
                val_acc_q8 += compute_accuracy(q8_logits, ss8)
                val_acc_q3 += compute_accuracy(q3_logits, ss3)
                val_sov_q3 += compute_sov_q3(q3_logits, ss3)
        
        val_loss /= len(val_loader)
        val_acc_q8 /= len(val_loader)
        val_acc_q3 /= len(val_loader)
        val_sov_q3 /= len(val_loader)
        
        print(f'Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}')
        print(f'Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}')
        print(f'Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}, Train SOV Q3={train_sov_q3:.4f}, Val SOV Q3={val_sov_q3:.4f}')
        
        # Early stopping check
        if val_acc_q8 > best_val_acc_q8:
            best_val_acc_q8 = val_acc_q8
            epochs_no_improve = 0
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state, f'best_rnn_scratch_fold{fold_idx+1}.pt')
            print(f"✓ New best model saved for fold {fold_idx+1}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
    
    fold_results.append({
        'fold': fold_idx + 1,
        'best_val_acc_q8': best_val_acc_q8,
        'train_acc_q8': train_acc_q8,
        'train_acc_q3': train_acc_q3,
        'val_acc_q3': val_acc_q3,
        'val_sov_q3': val_sov_q3
    })
    
    print(f"\nFold {fold_idx + 1} Best Val Acc Q8: {best_val_acc_q8:.4f}")

# Summary of all folds
print(f"\n{'='*60}")
print("K-Fold Cross-Validation Summary")
print(f"{'='*60}")
for result in fold_results:
    print(f"Fold {result['fold']}: Val Acc Q8 = {result['best_val_acc_q8']:.4f}")

avg_val_acc = sum([r['best_val_acc_q8'] for r in fold_results]) / n_folds
print(f"\nAverage Validation Accuracy Q8: {avg_val_acc:.4f}")

# Select best fold
best_fold = max(fold_results, key=lambda x: x['best_val_acc_q8'])
print(f"\nBest Fold: {best_fold['fold']} with Val Acc Q8: {best_fold['best_val_acc_q8']:.4f}")

# Test with best fold model
print(f"\n{'='*60}")
print("Testing with Best Fold Model")
print(f"{'='*60}")

model_test = ProteinRNN(vocab_size=vocab_size, embed_dim=embedding_dim).to(device)
model_test.load_state_dict(torch.load(f'best_rnn_scratch_fold{best_fold["fold"]}.pt', map_location=device))
if torch.cuda.device_count() > 1:
    model_test = nn.DataParallel(model_test)

model_test.eval()
test_loss = test_acc_q8 = test_acc_q3 = test_sov_q3 = 0.0
with torch.no_grad():
    for seqs, ss8, ss3 in test_loader:
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        lengths = (seqs != seq_vocab['<pad>']).sum(dim=1).to(torch.int64).cpu()
        q8_logits, q3_logits = model_test(seqs, lengths=lengths)
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)
        test_sov_q3 += compute_sov_q3(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)
test_sov_q3 /= len(test_loader)

print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy Q8: {test_acc_q8:.4f}')
print(f'Test Accuracy Q3: {test_acc_q3:.4f}')
print(f'Test SOV Q3: {test_sov_q3:.4f}')
